# 🚀 CodeBERT Text-Only Training Notebook

This notebook implements a streamlined, reproducible training pipeline for **CodeBERT (text-only)** — no DFG augmentation.

In [1]:
!pip install torch transformers scikit-learn tqdm

In [2]:
import os
import json
import torch
import logging
import random
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler, Subset
from torch.optim import AdamW
from transformers import (
    get_linear_schedule_with_warmup,
    RobertaConfig, RobertaModel, AutoTokenizer
)
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)
from collections import Counter

# --- CONFIGURATION ---
class Args:
    output_dir = "saved_models_codebert_text"
    model_name_or_path = "microsoft/codebert-base"

    # Data Path
    train_file = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"

    # Hyperparameters
    code_length = 384
    train_batch_size = 16
    eval_batch_size = 32
    learning_rate = 2e-5
    max_grad_norm = 1.0
    num_train_epochs = 5
    early_stopping_patience = 2
    seed = 42

    # Split ratios
    test_ratio = 0.10
    val_ratio = 0.08

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = Args()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(args.seed)

In [3]:
class SimpleModel(nn.Module):   
    def __init__(self, encoder, config):
        super(SimpleModel, self).__init__()
        self.encoder = encoder
        self.config = config
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, attention_mask=None, labels=None): 
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0] # [Batch, Seq, Hidden]
        
        # Use CLS token for classification
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob = F.softmax(logits, dim=-1)

        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            return loss, prob
        return prob

In [4]:
class SimpleDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args = args
        self.tokenizer = tokenizer

        with open(file_path, "r", encoding="utf-8") as f:
            # Raw lines only - parsed lazily in __getitem__. Storing parsed
            # entries here OOMs the kernel: each carries a large `dfg` array,
            # and DataLoader(num_workers=2) forks, so copy-on-write refcount
            # touching duplicates the whole structure per worker.
            self.lines = f.readlines()



    def __len__(self):
        return len(self.lines)

    def __getitem__(self, item):
        entry = json.loads(self.lines[item])

        code = entry.get("code", "")
        label = int(entry.get("label", 0)) if entry.get("label") is not None else 0

        tokens_obj = self.tokenizer(
            code,
            max_length=self.args.code_length,
            truncation=True,
            padding="max_length"
        )

        return {
            "input_ids": torch.tensor(tokens_obj["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(tokens_obj["attention_mask"], dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long)
        }

In [5]:
tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path)
full_dataset = SimpleDataset(tokenizer, args, args.train_file)

from collections import defaultdict
import math
import os
import numpy as np
def load_source_keys(filepath):
    """Stream the file and keep ONLY each entry's source key.

    The previous version parsed all 199,960 entries into memory a SECOND time
    (SimpleDataset already held a copy), which OOM'd the kernel before epoch 1.
    Nothing downstream needs the full entries - the split groups by source key,
    and this corpus has none, so every value is "unknown".
    """
    keys = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            e = json.loads(line)
            keys.append(infer_source(e))
            del e
    return keys

def infer_source(entry):
    for key in ("source", "dataset", "origin", "project"):
        value = entry.get(key)
        if value is not None and str(value).strip() != "":
            return str(value).strip()
    return "unknown"

def allocate_counts(total_needed, groups, fraction):
    raw = {g: len(v) * fraction for g, v in groups.items()}
    base = {g: int(math.floor(v)) for g, v in raw.items()}
    remainder = total_needed - sum(base.values())
    order = sorted(groups.keys(), key=lambda g: (raw[g] - base[g], len(groups[g])), reverse=True)
    for g in order[:remainder]:
        base[g] += 1
    return base

def stratified_three_way_split(source_keys, test_ratio=0.10, val_ratio=0.08, seed=42):
    rng = random.Random(seed)
    source_to_indices = defaultdict(list)
    for idx, key in enumerate(source_keys):
        source_to_indices[key].append(idx)

    for indices in source_to_indices.values():
        rng.shuffle(indices)

    total = len(source_keys)
    target_test = int(round(total * test_ratio))
    target_val = int(round(total * val_ratio))
    target_train = total - target_test - target_val

    test_alloc = allocate_counts(target_test, source_to_indices, test_ratio)
    trainval_groups = {}
    test_indices = []
    for source, indices in source_to_indices.items():
        take = min(test_alloc[source], len(indices))
        test_indices.extend(indices[:take])
        trainval_groups[source] = indices[take:]

    adjusted_val_ratio = val_ratio / (1.0 - test_ratio)
    val_alloc = allocate_counts(target_val, trainval_groups, adjusted_val_ratio)

    val_indices, train_indices = [], []
    for source, indices in trainval_groups.items():
        take = min(val_alloc[source], len(indices))
        val_indices.extend(indices[:take])
        train_indices.extend(indices[take:])

    train_indices = sorted(train_indices)
    val_indices = sorted(val_indices)
    test_indices = sorted(test_indices)

    assert len(train_indices) == target_train
    assert len(val_indices) == target_val
    assert len(test_indices) == target_test

    return train_indices, val_indices, test_indices

source_keys = load_source_keys(args.train_file)
assert len(source_keys) == len(full_dataset), "Dataset size mismatch"

train_indices, val_indices, test_indices = stratified_three_way_split(
    source_keys,
    test_ratio=args.test_ratio,
    val_ratio=args.val_ratio,
    seed=args.seed,
)

os.makedirs(args.output_dir, exist_ok=True)
np.save(os.path.join(args.output_dir, 'test_indices.npy'), np.array(test_indices))

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)
test_dataset = Subset(full_dataset, test_indices)

print("Dataset Split Results:")
print(f"- Total   : {len(full_dataset)}")
print(f"- Train   : {len(train_dataset)}")
print(f"- Val     : {len(val_dataset)}")
print(f"- Test    : {len(test_dataset)}")

def print_source_distribution(name, indices):
    from collections import Counter
    counts = Counter(source_keys[i] for i in indices)
    total = len(indices)
    print(f"\n{name} source distribution:")
    for src, cnt in sorted(counts.items()):
        print(f"  - {src}: {cnt} ({cnt / total:.2%})")

print_source_distribution("Train", train_indices)
print_source_distribution("Val", val_indices)
print_source_distribution("Test", test_indices)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/vocab.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/vocab.json "HTTP/1.1 200 OK"


...

merges.txt: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Dataset Split Results:
- Total   : 199960
- Train   : 163967
- Val     : 15997
- Test    : 19996

Train source distribution:
  - unknown: 163967 (100.00%)

Val source distribution:
  - unknown: 15997 (100.00%)

Test source distribution:
  - unknown: 19996 (100.00%)


In [6]:
def evaluate(model, dataset, args, tag="Eval"):
    dataloader = DataLoader(
        dataset,
        sampler=SequentialSampler(dataset),
        batch_size=args.eval_batch_size,
        num_workers=2,
        pin_memory=True
    )

    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {tag}"):
            probs = model(
                input_ids=batch["input_ids"].to(args.device),
                attention_mask=batch["attention_mask"].to(args.device)
            )
            all_probs.append(probs.detach().cpu().numpy())
            all_labels.extend(batch["label"].cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.array(all_labels)
    all_preds = np.argmax(all_probs, axis=-1)

    acc = accuracy_score(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs[:, 1])
    pr_auc = average_precision_score(all_labels, all_probs[:, 1])
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

    print("\n" + "=" * 40)
    print(f"RESULTS ({tag})")
    print("=" * 40)
    print(f"Accuracy : {acc:.4%}")
    print(f"ROC-AUC  : {roc_auc:.4f}")
    print(f"PR-AUC   : {pr_auc:.4f}")
    print(f"FN Count : {fn}  (missed vulnerabilities)")
    print(f"FP Count : {fp}  (false alarms)")
    print("-" * 40)
    print(classification_report(all_labels, all_preds, target_names=["Safe", "Vuln"], digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    metrics = {
        "accuracy": acc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "fn": fn,
        "fp": fp
    }
    return metrics, all_probs, all_labels


def train(model, train_dataset, val_dataset, args):
    train_dataloader = DataLoader(
        train_dataset,
        sampler=RandomSampler(train_dataset),
        batch_size=args.train_batch_size,
        num_workers=2,
        pin_memory=True
    )

    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=len(train_dataloader) * args.num_train_epochs
    )
    scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

    os.makedirs(args.output_dir, exist_ok=True)
    best_model_path = os.path.join(args.output_dir, "best_model.bin")

    best_val_acc = -1.0
    best_epoch = -1
    patience_counter = 0
    history = []

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0.0
        bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{args.num_train_epochs}")

        for step, batch in enumerate(bar):
            optimizer.zero_grad()

            with autocast('cuda'):
                loss, _ = model(
                    input_ids=batch["input_ids"].to(args.device),
                    attention_mask=batch["attention_mask"].to(args.device),
                    labels=batch["label"].to(args.device)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            tr_loss += loss.item()
            bar.set_postfix(loss=tr_loss / (step + 1))

        avg_train_loss = tr_loss / len(train_dataloader)
        print(f"\nEpoch {epoch + 1} training loss: {avg_train_loss:.6f}")

        val_metrics, _, _ = evaluate(model, val_dataset, args, tag=f"Validation Epoch {epoch + 1}")
        val_acc = val_metrics["accuracy"]

        history.append({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_accuracy": val_acc,
            "val_roc_auc": val_metrics["roc_auc"],
            "val_pr_auc": val_metrics["pr_auc"]
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved to {best_model_path} with val acc {best_val_acc:.4%}")
        else:
            patience_counter += 1
            print(f"No validation improvement. Patience {patience_counter}/{args.early_stopping_patience}")

            if patience_counter >= args.early_stopping_patience:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

    print(f"Best validation accuracy: {best_val_acc:.4%} at epoch {best_epoch}")

    return {
        "best_model_path": best_model_path,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "history": history
    }

In [7]:
# Initialization
config = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model = SimpleModel(encoder, config)
model.to(args.device)

# Train with validation-based early stopping
train_info = train(model, train_dataset, val_dataset, args)

# Load best-validation checkpoint
best_model_path = train_info["best_model_path"]
model.load_state_dict(torch.load(best_model_path, map_location=args.device))
print(f"Loaded best checkpoint from: {best_model_path}")

# Final test evaluation on the held-out stratified test set
test_metrics, probs, labels = evaluate(model, test_dataset, args, tag="CodeBERT Test")

# Save raw outputs for downstream analysis
np.save("/kaggle/working/codebert_text_train_probs.npy", probs)
np.save("/kaggle/working/codebert_text_train_labels.npy", labels)

# Write summary metrics to text file
preds = np.argmax(probs, axis=-1)
tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

out_path = "/kaggle/working/codebert_text_results.txt"
with open(out_path, "w") as f:
    f.write("Split        : 82/8/10 train/val/test (random shuffle, seed 42)\n")
    f.write("               NOTE: not source-stratified - the corpus has no source\n")
    f.write("               key, so infer_source() returns 'unknown' for every entry.\n")
    f.write("Test set     : unfiltered (19,996). Table 1 must be taken from test-2,\n")
    f.write("               which scores all six models on the duplicate-filtered set.\n")
    f.write(f"Seed         : {args.seed}\n")
    f.write(f"Max Epochs   : {args.num_train_epochs}\n")
    f.write(f"Patience     : {args.early_stopping_patience}\n")
    f.write(f"Best Epoch   : {train_info['best_epoch']}\n")
    f.write(f"Best Val Acc : {train_info['best_val_acc']:.4%}\n")
    f.write(f"Accuracy     : {test_metrics['accuracy']:.4%}\n")
    f.write(f"ROC-AUC      : {test_metrics['roc_auc']:.4f}\n")
    f.write(f"PR-AUC       : {test_metrics['pr_auc']:.4f}\n")
    f.write(f"FN           : {fn}\n")
    f.write(f"FP           : {fp}\n")

print(f"Saved results to {out_path}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/model.safetensors.index.

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/commits/refs%2Fpr%2F9 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/xet-read-token/99d7ef814601faaf7bdc2f774ffa7dade4f4d828 "HTTP/1.1 200 OK"


...


Epoch 5 training loss: 0.171170


Evaluating Validation Epoch 5: 100%|██████████| 500/500 [07:03<00:00,  1.18it/s]



RESULTS (Validation Epoch 5)
Accuracy : 88.3103%
ROC-AUC  : 0.9553
PR-AUC   : 0.9553
FN Count : 932  (missed vulnerabilities)
FP Count : 938  (false alarms)
----------------------------------------
              precision    recall  f1-score   support

        Safe     0.8830    0.8823    0.8827      7972
        Vuln     0.8832    0.8839    0.8835      8025

    accuracy                         0.8831     15997
   macro avg     0.8831    0.8831    0.8831     15997
weighted avg     0.8831    0.8831    0.8831     15997

Confusion Matrix:
[[7034  938]
 [ 932 7093]]
New best model saved to saved_models_codebert_text/best_model.bin with val acc 88.3103%
Best validation accuracy: 88.3103% at epoch 5
Loaded best checkpoint from: saved_models_codebert_text/best_model.bin


Evaluating CodeBERT Test: 100%|██████████| 625/625 [08:48<00:00,  1.18it/s]


RESULTS (CodeBERT Test)
Accuracy : 88.2476%
ROC-AUC  : 0.9584
PR-AUC   : 0.9597
FN Count : 1217  (missed vulnerabilities)
FP Count : 1133  (false alarms)
----------------------------------------
              precision    recall  f1-score   support

        Safe     0.8787    0.8862    0.8824      9953
        Vuln     0.8862    0.8788    0.8825     10043

    accuracy                         0.8825     19996
   macro avg     0.8825    0.8825    0.8825     19996
weighted avg     0.8825    0.8825    0.8825     19996

Confusion Matrix:
[[8820 1133]
 [1217 8826]]
Saved results to /kaggle/working/codebert_text_results.txt
